# 作者原版 Eval 解析问题复现

这个 notebook 用来**完全复刻作者 eval 里的关键解析路径**，并验证一个问题：

- 作者 `generate_response()` 返回的是 `prompt + generated answer` 的整段文本；
- 作者 `metrics_utils.generate_metrics()` 对整段 `response` 调用 `parse_open_lines(response)`；
- 因为 prompt 里本来就有 `Open Lines=[...]`，所以 graph metrics 可能解析到输入里的 existing open lines，而不是模型输出。

注意：这里会直接 import `LLM4DistReconfig/Dataset-Notebooks/utils` 里的作者源码函数。

In [ ]:
from pathlib import Path
import sys
import os
import contextlib
import io

# 如果 notebook 从 repo root 启动，cwd 就是 RL4DistReconfig。
# 如果从 tests/ 目录启动，则自动回到上一级。
cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "LLM4DistReconfig").exists() else cwd.parent
UPSTREAM_UTILS = REPO_ROOT / "LLM4DistReconfig" / "Dataset-Notebooks" / "utils"

assert UPSTREAM_UTILS.exists(), f"找不到作者 utils: {UPSTREAM_UTILS}"
sys.path.insert(0, str(UPSTREAM_UTILS))

print("REPO_ROOT =", REPO_ROOT)
print("作者 utils =", UPSTREAM_UTILS)

In [ ]:
# 这些 import 全部来自 LLM4DistReconfig/Dataset-Notebooks/utils，也就是作者代码。
from dataset_utils import prepare_train_data
from generation_utils import formatted_prompt, extract_output_data, generate_response
from metrics_utils import generate_metrics
from model_utils import (
    parse_open_lines,
    parse_available_lines,
    get_output_graph_edges,
    compute_cycles_loss,
    compute_invalid_edges_loss,
    compute_subgraphs_loss,
    get_model,
    get_tokenizer,
)

print("作者函数已导入")

## 1. 读取作者旧格式数据

这里用 `Dataset/Processed/train_33_nodes.csv`，因为 33 节点训练已经完成，且输出较短，便于肉眼检查。

In [ ]:
DATA_PATH = REPO_ROOT / "Dataset" / "Processed" / "train_33_nodes.csv"
train_ds, val_ds, test_ds = prepare_train_data(str(DATA_PATH))

print("train/val/test =", len(train_ds), len(val_ds), len(test_ds))
row = test_ds[0]
print("\n--- prompt 前 800 字符 ---")
print(row["prompt"][:800])
print("\n--- ground truth output 前 400 字符 ---")
print(row["output"][:400])

## 2. 用 fake answer 复现解析污染

这里我们不加载模型，直接构造：

```text
response = 作者格式 prompt + 一个明显不同的 fake answer
```

如果作者 `parse_open_lines(response)` 返回 prompt 里的 `Open Lines`，而不是 fake answer 里的 `Open Lines`，就说明 graph metrics 的输入被污染。

In [ ]:
raw_prompt = row["prompt"]
author_prompt = formatted_prompt(raw_prompt)

# fake answer 故意设置成和 prompt 里的 existing open lines 完全不同。
fake_answer = (
    "Output: Open Lines=[(1, 2), (2, 3), (3, 4), (4, 5), (5, 6)], "
    "Node Voltages=[0.1, 0.2], System Loss=999.999\n"
)
response = author_prompt + fake_answer

print("prompt 里的 Open Lines:")
print(parse_open_lines(author_prompt))

print("\nfake answer 里的 Open Lines:")
print(parse_open_lines(fake_answer))

print("\n作者 parse_open_lines(prompt + fake_answer) 的结果:")
print(parse_open_lines(response))

print("\n作者 extract_output_data(prompt + fake_answer) 的结果:")
parsed = extract_output_data(response)
print(parsed)

上面这个测试通常会出现一个关键现象：

- `parse_open_lines(prompt + fake_answer)` 返回的是 **prompt/input 里的 existing open lines**；
- `extract_output_data(prompt + fake_answer)` 往往返回的是 **fake answer 里的 Output**。

所以作者 CSV 里的 generated fields 可能来自模型输出，但 `Average Cycles / Invalid Edges / Subgraphs` 这条 graph metrics 路径会被 prompt 里的 input open lines 污染。

## 3. 直接用 input open lines 计算作者 graph metrics baseline

这个 cell 完全不跑模型，只解析每条 test prompt 里的 `Open Lines`，然后按作者代码计算 cycles / invalid edges / subgraphs。

如果这个 baseline 和作者 eval 输出的 graph metrics 一样，就说明作者 eval 的 graph metrics 实际上在算 input，而不是模型输出。

In [ ]:
def author_graph_metrics_from_text(text: str):
    """完全按作者 metrics_utils.generate_metrics 里的 graph 路径计算。"""
    predicted_lines = parse_open_lines(text)
    available_lines = parse_available_lines(text)
    graph_edges = get_output_graph_edges(predicted_lines, available_lines)
    # compute_subgraphs_loss 里有作者 debug print，这里重定向掉，避免刷屏。
    with contextlib.redirect_stdout(io.StringIO()):
        cycles = compute_cycles_loss(graph_edges)
        invalid = compute_invalid_edges_loss(predicted_lines, available_lines)
        subgraphs = compute_subgraphs_loss(graph_edges)
    return float(cycles), float(invalid), float(subgraphs)

def input_only_baseline(dataset):
    total_c = total_i = total_s = 0.0
    for item in dataset:
        text = formatted_prompt(item["prompt"])
        c, i, s = author_graph_metrics_from_text(text)
        total_c += c
        total_i += i
        total_s += s
    n = len(dataset)
    return {
        "n": n,
        "avg_cycles": total_c / n,
        "avg_invalid_edges": total_i / n,
        "avg_subgraphs": total_s / n,
    }

baseline = input_only_baseline(test_ds)
baseline

如果你已经跑过 `Eval/reproduce_author_eval.py`，可以把作者复刻 eval 的 `metrics.txt` 打开，对比上面的 baseline。

In [ ]:
metrics_path = REPO_ROOT / "outputs" / "author_reproduce_eval" / "sft_author_strict_custom_ep10_33_n500" / "metrics.txt"
if metrics_path.exists():
    print(metrics_path)
    print(metrics_path.read_text())
else:
    print("还没有找到 metrics.txt:", metrics_path)

## 3.1 统计 33 nodes 的 GT output open lines 数量

这里统计数据集 `output` 里的 ground-truth `Open Lines` 一般有几条。

对于 IEEE 33-bus 系统，原始 `Lines` 数通常是 37，径向闭合边需要 `33 - 1 = 32` 条，所以 open lines 数应为 `37 - 32 = 5`。

这个统计也有助于理解作者 eval 里为什么 `Average Invalid Edges` 会稳定等于 5：作者 graph metric 路径先解析 input open lines，然后 `get_output_graph_edges()` 原地 append 反向边，反向边又被 `compute_invalid_edges_loss()` 当成 invalid。

In [ ]:
from collections import Counter

def count_output_open_lines(dataset):
    counter = Counter()
    for item in dataset:
        # 这里直接用作者 parse_open_lines 解析 output 字段。
        open_lines = parse_open_lines(item["output"])
        counter[len(open_lines)] += 1
    return dict(sorted(counter.items()))

open_line_count_by_split = {
    "train": count_output_open_lines(train_ds),
    "validation": count_output_open_lines(val_ds),
    "test": count_output_open_lines(test_ds),
}
open_line_count_by_split

In [ ]:
# 汇总全量 33 nodes 数据。
all_counter = Counter()
for ds in [train_ds, val_ds, test_ds]:
    all_counter.update(count_output_open_lines(ds))
dict(sorted(all_counter.items()))

## 4. 看几条真实模型生成到底是什么

下面这个 cell 会加载 base model + adapter，并调用作者 `generate_response()`。它可能比较慢，也需要 GPU。

默认用 33 节点 custom-loss ep10 的 `final_adapter`。如果你要测 69/84/混合，改 `ADAPTER_PATH` 和 `DATA_PATH` 即可。

In [ ]:
# 按需修改这三个路径。
MODEL_ID = str(REPO_ROOT / ".." / "models" / "meta-llama" / "Llama-3.1-8B-Instruct")
ADAPTER_PATH = str(REPO_ROOT / "runs" / "llama31_8b_instruct" / "sft_author_strict_custom_ep10__on__train_33_nodes" / "final_adapter")
MAX_NEW_TOKENS = 1400

print("MODEL_ID =", MODEL_ID)
print("ADAPTER_PATH =", ADAPTER_PATH)
print("存在 adapter?", Path(ADAPTER_PATH).exists())

In [ ]:
# 这个 cell 会加载模型，耗显存。只在需要查看真实输出时运行。
from generation_utils import peft_merge_unload

model = peft_merge_unload(MODEL_ID, ADAPTER_PATH)
tokenizer = get_tokenizer(MODEL_ID)
print("模型和 tokenizer 已加载")

In [ ]:
# 查看几条真实作者 generate_response 的 raw response，以及两个 parser 分别解析到了什么。
sample_indices = [0, 1, 2]

for idx in sample_indices:
    item = test_ds[idx]
    print("=" * 100)
    print("test index:", idx)
    print("GT output:", item["output"][:300].replace("\n", "\\n"))
    response, output_time = generate_response(
        user_input=item["prompt"],
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=MAX_NEW_TOKENS,
    )
    print("生成耗时:", output_time)
    print("\n--- raw response 前 1500 字符 ---")
    print(response[:1500])
    print("\n--- raw response 后 800 字符 ---")
    print(response[-800:])
    print("\nparse_open_lines(raw response) ->")
    print(parse_open_lines(response))
    print("\nextract_output_data(raw response) ->")
    print(extract_output_data(response))

## 5. 小规模调用作者 generate_metrics

这个 cell 会真正调用作者 `generate_metrics()`，例如抽 5 条，输出格式和作者 eval 一样。

注意：作者 eval 会随机抽样、单条 HF generate、并打印很多 debug 信息，所以会慢且刷屏。

In [ ]:
from functools import partial

small_out_dir = REPO_ROOT / "outputs" / "author_eval_bug_notebook_small"
small_out_dir.mkdir(parents=True, exist_ok=True)

response_fn = partial(
    generate_response,
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
)

generate_metrics(
    test_ds,
    response_fn,
    num_samples=5,
    filename_txt=str(small_out_dir / "metrics.txt"),
    filename_csv=str(small_out_dir / "responses.csv"),
)

print((small_out_dir / "metrics.txt").read_text())

## 结论

这个 notebook 要验证的核心点是：

1. 作者 `generate_response()` 的 response 包含 prompt；
2. 作者 `parse_open_lines(response)` 会先命中 prompt 里的 `Open Lines=[existing_open_lines]`；
3. 因此作者 eval 的 `Average Cycles / Average Invalid Edges / Average Subgraphs` 很可能是在评估 input existing open lines，而不是模型输出；
4. `extract_output_data(response)` 通常能解析到模型的 `Output:`，所以 CSV 里的 generated fields 和 graph metrics 可能来自不同来源；
5. 如果 input-only baseline 与作者 eval 的 graph metrics 一致，就能实锤 graph metrics 被 input 污染。